# Metaclonotype API Examples

This notebook demonstrates how to build and analyze metaclonotypes as a lightweight clustering layer over `LocusRepertoire`.

In [ ]:
# Print key environment versions for reproducibility
import platform
import polars as pl
import mir

print('python', platform.python_version())
print('polars', pl.__version__)
print('mir', getattr(mir, '__version__', 'unknown'))

In [ ]:
# Build a tiny repertoire and connected-component metaclonotypes
from mir.common.clonotype import Clonotype
from mir.common.repertoire import LocusRepertoire
from mir.common.metaclonotype import (
    metaclonotypes_from_components,
    summarize_metaclonotypes,
    functional_diversity,
    functional_hill_curve,
)

rep = LocusRepertoire(
    [
        Clonotype(sequence_id='c1', locus='TRB', junction_aa='CASSLGQETQYF', v_gene='TRBV5-1*01', j_gene='TRBJ2-7*01', duplicate_count=10, umi_count=4, _validate=False),
        Clonotype(sequence_id='c2', locus='TRB', junction_aa='CASSLGQETQFF', v_gene='TRBV5-1*01', j_gene='TRBJ2-7*01', duplicate_count=5, umi_count=3, _validate=False),
        Clonotype(sequence_id='c3', locus='TRB', junction_aa='CATSLGQETQYF', v_gene='TRBV5-1*01', j_gene='TRBJ2-7*01', duplicate_count=2, umi_count=1, _validate=False),
    ],
    locus='TRB',
)

meta = metaclonotypes_from_components([["c1", "c2"], ["c3"]])
rep.set_metaclonotypes(meta)

summary = summarize_metaclonotypes(rep, meta)
summary

In [ ]:
# Compute functional diversity metrics from metaclonotype counts
func_div = functional_diversity(rep, meta, count_field='duplicate_count')
print(func_div)

functional_hill_curve(rep, meta, count_field='duplicate_count', q_values=[0.0, 1.0, 2.0])

In [ ]:
# Build metaclonotypes from ALICE/TCRNET metadata-marked representatives
from mir.biomarkers.alice import metaclonotypes_from_alice
from mir.biomarkers.tcrnet import metaclonotypes_from_tcrnet

rep.clonotypes[0].clone_metadata['alice_q_value'] = 0.01
rep.clonotypes[0].clone_metadata['tcrnet_q_value'] = 0.01

alice_meta = metaclonotypes_from_alice(rep, q_value_max=0.05)
tcrnet_meta = metaclonotypes_from_tcrnet(rep, q_value_max=0.05)

print('alice clusters', alice_meta.n_clusters)
print('tcrnet clusters', tcrnet_meta.n_clusters)